In [1]:
from pathlib import Path

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [2]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
test_loader = build_loader_for_model(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=(
        str(PROJECT_ROOT / experiment_config.resume_from)
        if experiment_config.resume_from is not None
        else None
    ),
    train_loader=train_loader,
    valid_loader=val_loader,
)

if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from

    # Load pretrained weights, but NOT the old Adam optimizer state.
    trainer.load_checkpoint(
        resume_from,
        load_optimizer=True,
    )

    # Set the fine-tuning learning rate.
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    # Start a new experiment.
    # trainer.start_epoch = 0
    # trainer.best_val_loss = float("inf")

    # print("Fine-tuning from:", resume_from)
    print("Learning rate:", hyperparams["learning_rate"])
    # print("Optimizer state: reset")

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\latest.pt (epoch 3, best val loss 0.1532)
Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\latest.pt (epoch 3, best val loss 0.1532)
Learning rate: 0.0001


In [3]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Validation: 100%|██████████| 151/151 [00:26<00:00,  5.68it/s]



Epoch 4/20

TRAIN
Loss   | Total: 0.1243 | NP: 0.0094 | NS: 0.0584
Onset  | P: 0.8311 | R: 0.8349 | F1: 0.8330
Action | P: 0.9089 | R: 0.8804 | F1: 0.8945
Active Exact Accuracy: 0.3956
GT Active Rate: 4.99% | Predicted Active Rate: 4.84%

VAL
Loss   | Total: 0.1533 | NP: 0.0134 | NS: 0.0592
Onset  | P: 0.7838 | R: 0.7002 | F1: 0.7396
Action | P: 0.8668 | R: 0.8353 | F1: 0.8508
Active Exact Accuracy: 0.3729
GT Active Rate: 4.37% | Predicted Active Rate: 4.21%


Epochs:   6%|▌         | 1/17 [09:02<2:24:32, 542.02s/it]

Saved checkpoint: checkpoints\latest.pt


Epochs:  12%|█▏        | 2/17 [18:16<2:17:18, 549.25s/it]

... (10 outputs left out)


Epoch 10/20

TRAIN
Loss   | Total: 0.1091 | NP: 0.0082 | NS: 0.0514
Onset  | P: 0.8494 | R: 0.8569 | F1: 0.8531
Action | P: 0.9184 | R: 0.9003 | F1: 0.9093
Active Exact Accuracy: 0.4279
GT Active Rate: 4.99% | Predicted Active Rate: 4.89%

VAL
Loss   | Total: 0.1603 | NP: 0.0145 | NS: 0.0585
Onset  | P: 0.7525 | R: 0.7457 | F1: 0.7491
Action | P: 0.8748 | R: 0.8388 | F1: 0.8564
Active Exact Accuracy: 0.3785
GT Active Rate: 4.37% | Predicted Active Rate: 4.19%
Saved checkpoint: checkpoints\latest.pt


Epochs:  41%|████      | 7/17 [1:06:12<1:34:34, 567.43s/it]


epoch,▁▂▃▅▆▇█
time/epoch_seconds,▆█▄▆▂▁▅
time/train_seconds,▆█▄▅▂▁▅
time/val_seconds,█▄▄▇▁▁▃
train/action_f1,▁▃▄▅▆▇█
train/action_precision,▁▂▄▅▆▇█
train/action_recall,▁▃▄▅▆▇█
train/active_exact_accuracy,▁▃▄▅▆▇█
train/gt_active_rate,▅▆█▃▄▁▃
train/np_loss,█▆▅▄▃▂▁
+18,...


KeyboardInterrupt: 